# Notebook 03 — Dual Clustering for the Definition of Expert Domains

---

## Objective

Define the domains of the Mixture-of-Experts (MoE) by clustering hourly electricity demand of Colombia's National Interconnected System (SIN) independently along two time scales:

- **TYPE_DAY** — clustering of **daily profiles** (24 hourly values per calendar day);
- **TYPE_WEEK** — clustering of **weekly profiles** (168 hourly values per ISO week).

Every hourly observation then receives a day cluster (`cluster_day`) and a week cluster (`cluster_week`). The Cartesian product of the two partitions defines the expert domains, identified as `expert_id = "day{cluster_day}_week{cluster_week}"`, so the number of potential experts is $k_{day} \times k_{week}$.

## Inputs

| File | Produced by | Content |
|---|---|---|
| `data/processed/features.parquet` | Notebook 02 (preprocessing / feature engineering) | Hourly demand series and engineered features. Only the demand (target) column is used here. |

## Outputs

All clustering artifacts are written to `data/clustering/` and all figures to `results/`.

| File | Content | Used by |
|---|---|---|
| `data/clustering/cluster_assignments.parquet` | One row per hour: `datetime`, `cluster_day`, `cluster_week`, `expert_id` | Notebooks 04–08 |
| `data/clustering/centroids_day.pkl` | Centroids of the selected TYPE_DAY configuration (+ algorithm, k) | Notebook 06 (distance-based soft gating) |
| `data/clustering/centroids_week.pkl` | Centroids of the selected TYPE_WEEK configuration (+ algorithm, k) | Notebook 06 |
| `data/clustering/clustering_metrics.json` | Internal validation metrics for every (algorithm, k) evaluated | Reference only |
| `data/clustering/top_configs.json` | Top-3 configurations per time scale and the selected combination | Reference only |
| `data/clustering/top3_labels_archive.pkl` | Labels and centroids of the top-3 configurations per time scale | Reference only |
| `results/03_clustering_*.png` | Figures (profiles, frequency heatmap, calendar view, temporal consistency, metric comparison) | — |

## Place in the pipeline

01 data acquisition (XM S.A. SINERGOX API) → 02 preprocessing / feature engineering → **03 clustering of daily and weekly demand profiles** → 04 cluster analysis and cluster classifiers → 05 expert model selection per (day cluster, week cluster) → 06 gating design → 07 hyperparameter optimization (HPO) → 08 production assembly → 09 benchmark.

The cluster assignments produced here are the supervision targets of the classifiers trained in Notebook 04 and define the training subsets of the experts selected in Notebook 05.

## Environment setup

This notebook runs unchanged on a local machine (pip/venv or conda), in Docker and in Google Colab.

**`OUTPUT_ROOT`** (next cell) is the folder where the pipeline stores `data/`, `models/` and `results/`:

| Value | Where artifacts are stored |
|---|---|
| `None` (default) | The repository folder, or the folder in the `MOE_OUTPUT_ROOT` environment variable if it is set |
| `"/content/drive/MyDrive/MoEForecaster"` | Google Drive (Colab only; Drive is mounted automatically) |
| `"/any/absolute/path"` or `"relative/path"` | Any folder; relative paths are resolved from the repository root |

Use the **same `OUTPUT_ROOT` in every notebook**: each notebook reads the artifacts written by the previous ones. Setting the `MOE_OUTPUT_ROOT` environment variable once is the easiest way to keep it consistent.

The cell after it finds the repository, which it clones first when running in Colab. In Colab it also installs the pinned dependencies from `requirements.txt` and makes the `moe_forecaster` package in `src/` importable.

In [ ]:
# ============================================================
# PARAMETERS: the only cell you may need to edit
# ============================================================
# Folder for data/, models/ and results/ (see the table above).
# None -> repository folder, or $MOE_OUTPUT_ROOT if that variable is set.
OUTPUT_ROOT = None

In [ ]:
# ============================================================
# ENVIRONMENT BOOTSTRAP: no edits needed
# ============================================================
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/astorress/MoEForecaster.git"
IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start):
    """Walk up from `start` until the folder that contains src/moe_forecaster/."""
    for folder in [start, *start.parents]:
        if (folder / "src" / "moe_forecaster").is_dir():
            return folder
    return None


REPO_ROOT = find_repo_root(Path.cwd().resolve())

if REPO_ROOT is None and IN_COLAB:
    # Colab opens the notebook on its own, without the rest of the repository
    REPO_ROOT = Path("/content/MoEForecaster")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

if REPO_ROOT is None:
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the MoEForecaster folder "
        "(local, conda or Docker), or open the notebook in Google Colab."
    )

if IN_COLAB:
    # Install the pinned versions once per Colab runtime. Colab's preinstalled
    # PyTorch (built with CUDA) is kept, so the torch pin is skipped here.
    marker = Path("/content/.moe_requirements_installed")
    if not marker.exists():
        reqs = [
            line for line in (REPO_ROOT / "requirements.txt").read_text().splitlines()
            if line.strip() and not line.startswith(("#", "torch"))
        ]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
        marker.touch()
        print("Dependencies installed. If Colab asks to restart the session, restart it and run the notebook again.")

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from moe_forecaster.config import setup_paths

DATA_DIR, MODELS_DIR, RESULTS_DIR = setup_paths(OUTPUT_ROOT, repo_root=REPO_ROOT)

---
## 1. Imports and configuration

Library imports, global configuration (output folder for clustering artifacts, random seed) and the plotting style shared by all figures of this notebook.

In [ ]:
import os
import json
import pickle
import warnings
from itertools import product
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score

from tslearn.clustering import KShape, TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ============================================================
# CONFIGURATION
# ============================================================
CLUSTERING_DIR = os.path.join(DATA_DIR, "clustering")  # created by the setup cell
RANDOM_SEED = 42
N_JOBS = -1

# ── Plot style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 120,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "font.family": "serif",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

PALETTE = sns.color_palette("Set2", 10)

np.random.seed(RANDOM_SEED)
print("Configuration loaded.")
print(f"  DATA_DIR:       {os.path.abspath(DATA_DIR)}")
print(f"  CLUSTERING_DIR: {os.path.abspath(CLUSTERING_DIR)}")
print(f"  RANDOM_SEED:    {RANDOM_SEED}")

---
## 2. Data loading

The feature table produced by Notebook 02 is loaded and indexed by its hourly timestamp. Only the demand series (the target column) is needed for clustering. The target column is identified by name among a list of candidate names; if none is found, the first numeric column is used as a fallback.

The analysis period is then restricted to **2022-01-01 – 2025-12-31**, so that the clusters (and therefore the expert domains) are defined on this period only.

In [ ]:
features_path = os.path.join(DATA_DIR, "processed", "features.parquet")
df = pd.read_parquet(features_path)

# Ensure a DatetimeIndex
if not isinstance(df.index, pd.DatetimeIndex):
    if "datetime" in df.columns:
        df = df.set_index("datetime")
    elif "fecha" in df.columns:
        df = df.set_index("fecha")
    else:
        # Use the first column if it looks like a datetime
        first_col = df.columns[0]
        df[first_col] = pd.to_datetime(df[first_col])
        df = df.set_index(first_col)

df.index = pd.to_datetime(df.index)
df = df.sort_index()

# Identify the target (demand) column; the candidate names include legacy Spanish column names
target_candidates = [c for c in df.columns if c.lower() in [
    "demanda", "demand", "demanda_sin", "target", "load", "value"
]]
if target_candidates:
    TARGET_COL = target_candidates[0]
else:
    # Fallback: first numeric column
    TARGET_COL = df.select_dtypes(include=[np.number]).columns[0]

In [ ]:
print(f"Features loaded: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Target column: {TARGET_COL}")
print(f"Missing values in the target: {df[TARGET_COL].isna().sum()}")
df.head(3)

In [ ]:
# Restrict the analysis period
start_date = '2022-01-01'
end_date = '2025-12-31'

df = df.loc[start_date:end_date]

print(f"DataFrame filtered from {start_date} to {end_date}.")
print(f"New DataFrame shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")

---
## 3. Profile construction

Two types of profiles are built from the hourly demand series:

1. **Daily profiles (24-D)**: each calendar day is represented by the vector of its 24 hourly demand values. Days with fewer than 24 hourly records are discarded.
2. **Weekly profiles (168-D)**: each ISO week (ISO year + ISO week number) is represented by its 168 hourly values (Monday 00:00 to Sunday 23:00). Only complete weeks (exactly 168 hours) are kept, so partial weeks at the beginning and end of the period are discarded.

For each set, a **row-wise z-normalized** version is also computed (each profile minus its own mean, divided by its own standard deviation; constant profiles are guarded against division by zero). Row-wise z-normalization removes level differences (seasonality, economic growth) and keeps only the *shape* of the profile.

> **Note on what is clustered.** The clustering sweeps in Sections 4 and 5 are run on the **raw (non-normalized) profiles** `daily_matrix` and `weekly_matrix`; the normalized arrays `daily_normalized` and `weekly_normalized` are computed and inspected here but are not used by the clustering. The resulting clusters therefore reflect both the level and the shape of demand.

In [ ]:
# ── Daily profiles (24-D vectors) ─────────────────────────────────────────────
demand_series = df[TARGET_COL].copy()

daily_profiles = demand_series.groupby(demand_series.index.date).apply(
    lambda x: x.values if len(x) == 24 else None
)
# Drop days with incomplete data (!= 24 hours)
daily_profiles = daily_profiles.dropna()
daily_dates = pd.to_datetime(daily_profiles.index)
daily_matrix = np.stack(daily_profiles.values)  # (n_days, 24)

print(f"Daily profiles: {daily_matrix.shape} (days x hours)")
print(f"  Complete days: {daily_matrix.shape[0]}")
print(f"  Incomplete days dropped: {demand_series.index.normalize().nunique() - daily_matrix.shape[0]}")

# Row-wise z-normalization
daily_means = daily_matrix.mean(axis=1, keepdims=True)
daily_stds = daily_matrix.std(axis=1, keepdims=True)

# Avoid division by zero for constant days (if any)
daily_stds[daily_stds < 1e-8] = 1.0
daily_normalized = (daily_matrix - daily_means) / daily_stds

print(f"\nShape of normalized daily profiles: {daily_normalized.shape}")
print(f"  Row means ~ 0: {np.abs(daily_normalized.mean(axis=1)).max():.2e}")
print(f"  Row standard deviations ~ 1: {daily_normalized.std(axis=1).mean():.4f}")

In [ ]:
daily_matrix[0]

In [ ]:
daily_normalized[0]

In [ ]:
# ── Weekly profiles (168-D vectors) ───────────────────────────────────────────
# Group by ISO year-week and keep only complete weeks (168 hours)
demand_series_week = demand_series.copy()
demand_series_week = demand_series_week.to_frame(name="demand")
demand_series_week["iso_year"] = demand_series_week.index.isocalendar().year.values
demand_series_week["iso_week"] = demand_series_week.index.isocalendar().week.values
demand_series_week["year_week"] = (
    demand_series_week["iso_year"].astype(str) + "-W" +
    demand_series_week["iso_week"].astype(str).str.zfill(2)
)

weekly_groups = demand_series_week.groupby("year_week")["demand"]
weekly_profiles_dict = {}
weekly_start_dates = {}

for yw, group in weekly_groups:
    if len(group) == 168:
        weekly_profiles_dict[yw] = group.values
        weekly_start_dates[yw] = group.index.min()

weekly_labels = list(weekly_profiles_dict.keys())
weekly_matrix = np.stack(list(weekly_profiles_dict.values()))  # (n_weeks, 168)
weekly_dates = pd.to_datetime(list(weekly_start_dates.values()))

print(f"Weekly profiles: {weekly_matrix.shape} (weeks x hours)")
print(f"  Complete weeks: {weekly_matrix.shape[0]}")

# Row-wise z-normalization
weekly_means = weekly_matrix.mean(axis=1, keepdims=True)
weekly_stds = weekly_matrix.std(axis=1, keepdims=True)
weekly_stds[weekly_stds < 1e-8] = 1.0
weekly_normalized = (weekly_matrix - weekly_means) / weekly_stds

print(f"\nShape of normalized weekly profiles: {weekly_normalized.shape}")
print(f"  Row means ~ 0: {np.abs(weekly_normalized.mean(axis=1)).max():.2e}")
print(f"  Row standard deviations ~ 1: {weekly_normalized.std(axis=1).mean():.4f}")

In [ ]:
weekly_matrix[0]

In [ ]:
weekly_normalized[0]

In [ ]:
# ── Quick visual check: random sample of raw profiles and the mean profile ─────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Sample of daily profiles
ax = axes[0]
sample_idx = np.random.choice(len(daily_matrix), size=min(100, len(daily_matrix)), replace=False)
for i in sample_idx:
    ax.plot(range(24), daily_matrix[i], alpha=0.08, color="steelblue", linewidth=0.5)
ax.plot(range(24), daily_matrix.mean(axis=0), color="darkred", linewidth=2, label="Mean profile")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Demand")
ax.set_title("Daily Profiles (24h)")
ax.set_xticks(range(0, 24, 3))
ax.legend()

# Sample of weekly profiles
ax = axes[1]
sample_idx_w = np.random.choice(len(weekly_matrix), size=min(50, len(weekly_matrix)), replace=False)
for i in sample_idx_w:
    ax.plot(range(168), weekly_matrix[i], alpha=0.08, color="steelblue", linewidth=0.5)
ax.plot(range(168), weekly_matrix.mean(axis=0), color="darkred", linewidth=2, label="Mean profile")
ax.set_xlabel("Hour of week (Mon 0h to Sun 23h)")
ax.set_ylabel("Demand")
ax.set_title("Weekly Profiles (168h)")
day_ticks = [i * 24 for i in range(7)]
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
ax.set_xticks(day_ticks)
ax.set_xticklabels(day_labels)
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "03_clustering_profiles_overview.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Profile preparation complete.")

---
## 4. TYPE_DAY clustering (daily profiles, 24h)

### 4.1 Candidate algorithms and validation metrics

Three partitional clustering algorithms (all from `tslearn`) are compared for $k \in \{2, \dots, 8\}$ on the daily profiles:

| Key | Algorithm | Notes |
|---|---|---|
| `kmeans_euclidean` | `TimeSeriesKMeans` with Euclidean distance | 20 initializations |
| `kshape` | k-Shape (shape-based distance built on normalized cross-correlation) | single initialization |
| `kmeans_dtw` | `TimeSeriesKMeans` with Dynamic Time Warping (DTW) distance | 20 initializations |

The helper `run_clustering` also implements a soft-DTW k-means variant (`kmeans_softdtw`, $\gamma = 0.1$), but it is not included in the sweeps below.

Each configuration is evaluated with three internal validation indices computed by scikit-learn on the profiles with Euclidean distance: the **Silhouette score** (higher is better), the **Davies–Bouldin index** (lower is better) and the **Calinski–Harabasz index** (higher is better; reported but not used for ranking).

### 4.2 Selection criterion

Configurations are ranked separately by Silhouette (descending) and by Davies–Bouldin (ascending); the two ranks are summed into a **combined rank** (lower is better), following Chicco (2025). The top-3 configurations are retained, and the first one is the automatic candidate (see Section 6.1 for the final choice of $k$).

In [ ]:
def run_clustering(
    data_2d: np.ndarray,
    algorithm: str,
    k: int,
    random_seed: int = 42,
    max_iter: int = 50,
) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Run a single clustering configuration.

    Parameters
    ----------
    data_2d : np.ndarray of shape (n_samples, n_features)
        Profiles to cluster (one profile per row).
    algorithm : str
        One of: 'kmeans_euclidean', 'kshape', 'kmeans_dtw', 'kmeans_softdtw'
    k : int
        Number of clusters.
    random_seed : int
        Random seed for reproducibility.
    max_iter : int
        Maximum number of iterations for the tslearn algorithms.

    Returns
    -------
    labels : np.ndarray of shape (n_samples,)
    centroids : np.ndarray
    inertia : float
    """
    n_samples, n_features = data_2d.shape

    # 3-D view expected by tslearn: (n_samples, n_features, 1). Note: the models below are fit on data_2d.
    data_3d = data_2d.reshape(n_samples, n_features, 1)

    if algorithm == "kmeans_euclidean":
        model = TimeSeriesKMeans(
            n_clusters=k,
            random_state=random_seed,
            n_init=20,
            max_iter=max_iter,
            verbose=False,
            n_jobs=-1
        )
        labels = model.fit_predict(data_2d)
        centroids = model.cluster_centers_.squeeze()
        inertia = model.inertia_
        return labels, centroids, inertia

    if algorithm == "kshape":
        model = KShape(
            n_clusters=k,
            random_state=random_seed,
            max_iter=max_iter,
            verbose=False,
        )
        labels = model.fit_predict(data_2d)
        centroids = model.cluster_centers_.squeeze()  # (k, n_features)
        inertia = model.inertia_
        return labels, centroids, inertia

    elif algorithm == "kmeans_dtw":
        model = TimeSeriesKMeans(
            n_clusters=k,
            metric="dtw",
            random_state=random_seed,
            max_iter=max_iter,
            n_init=20,
            verbose=False,
            n_jobs=-1,
        )
        labels = model.fit_predict(data_2d)
        centroids = model.cluster_centers_.squeeze()
        inertia = model.inertia_
        return labels, centroids, inertia

    elif algorithm == "kmeans_softdtw":
        model = TimeSeriesKMeans(
            n_clusters=k,
            metric="softdtw",
            metric_params={"gamma": 0.1},
            random_state=random_seed,
            max_iter=max_iter,
            n_init=3,
            verbose=False,
        )
        labels = model.fit_predict(data_2d)
        centroids = model.cluster_centers_.squeeze()
        inertia = model.inertia_
        return labels, centroids, inertia

    else:
        raise ValueError(f"Unknown algorithm: {algorithm}")


def compute_clustering_metrics(
    data_2d: np.ndarray,
    labels: np.ndarray,
) -> Dict[str, float]:
    """
    Compute internal clustering validation metrics (Euclidean distance).
    """
    n_unique = len(np.unique(labels))
    if n_unique < 2:
        return {
            "silhouette": -1.0,
            "davies_bouldin": np.inf,
            "calinski_harabasz": 0.0,
        }

    return {
        "silhouette": silhouette_score(data_2d, labels),
        "davies_bouldin": davies_bouldin_score(data_2d, labels),
        "calinski_harabasz": calinski_harabasz_score(data_2d, labels),
    }


print("Clustering helper functions defined.")

In [ ]:
# ── TYPE_DAY sweep ────────────────────────────────────────────────────────
DAY_ALGORITHMS = ["kmeans_euclidean", "kshape", "kmeans_dtw"]
DAY_K_RANGE = list(range(2, 9))  # 2..8

day_results = []          # list of dicts for the results table
day_all_labels = {}       # (algo, k) -> labels
day_all_centroids = {}    # (algo, k) -> centroids

total_configs = len(DAY_ALGORITHMS) * len(DAY_K_RANGE)
print(f"Running {total_configs} TYPE_DAY configurations...\n")

for algo in DAY_ALGORITHMS:
    for k in DAY_K_RANGE:
        print(f"  {algo:20s} k={k} ...", end=" ", flush=True)
        try:
            labels, centroids, inertia = run_clustering(
                daily_matrix, algo, k, random_seed=RANDOM_SEED
            )
            metrics = compute_clustering_metrics(daily_matrix, labels)
            metrics["inertia"] = inertia
            metrics["algorithm"] = algo
            metrics["k"] = k
            metrics["clustering_type"] = "type_day"
            day_results.append(metrics)
            day_all_labels[(algo, k)] = labels
            day_all_centroids[(algo, k)] = centroids
            print(
                f"Sil={metrics['silhouette']:.4f}  "
                f"DB={metrics['davies_bouldin']:.4f}  "
                f"CH={metrics['calinski_harabasz']:.1f}"
            )
        except Exception as e:
            print(f"FAILED: {e}")
            day_results.append({
                "algorithm": algo, "k": k, "clustering_type": "type_day",
                "silhouette": np.nan, "davies_bouldin": np.nan,
                "calinski_harabasz": np.nan, "inertia": np.nan,
            })

day_results_df = pd.DataFrame(day_results)
print(f"\nCompleted {len(day_results)} configurations.")

In [ ]:
# ── TYPE_DAY results table ────────────────────────────────────────────────
display_cols = ["algorithm", "k", "silhouette", "davies_bouldin", "calinski_harabasz", "inertia"]
day_display = day_results_df[display_cols].copy()
day_display = day_display.sort_values(["algorithm", "k"]).reset_index(drop=True)

# Display formatting
styled = day_display.style.format({
    "silhouette": "{:.4f}",
    "davies_bouldin": "{:.4f}",
    "calinski_harabasz": "{:.1f}",
    "inertia": "{:.1f}",
}).background_gradient(
    subset=["silhouette"], cmap="Greens"
).background_gradient(
    subset=["davies_bouldin"], cmap="Reds_r"
)

print("TYPE_DAY clustering results:")
display(styled)

In [ ]:
# ── Automatic selection: combined Silhouette + DB ranking ───────────────
def rank_clustering_configs(results_df: pd.DataFrame, top_n: int = 3) -> pd.DataFrame:
    """
    Rank clustering configurations by Silhouette (higher = better)
    and Davies-Bouldin (lower = better), following Chicco (2025).

    Returns the top_n rows of a DataFrame sorted by combined rank.
    """
    df_rank = results_df.dropna(subset=["silhouette", "davies_bouldin"]).copy()

    # Ranks: silhouette descending (rank 1 = highest), DB ascending (rank 1 = lowest)
    df_rank["rank_silhouette"] = df_rank["silhouette"].rank(ascending=False).astype(int)
    df_rank["rank_db"] = df_rank["davies_bouldin"].rank(ascending=True).astype(int)
    df_rank["combined_rank"] = df_rank["rank_silhouette"] + df_rank["rank_db"]
    df_rank = df_rank.sort_values("combined_rank").reset_index(drop=True)

    return df_rank.head(top_n)


day_top3 = rank_clustering_configs(day_results_df, top_n=3)
print("TOP-3 TYPE_DAY configurations (combined Silhouette + DB rank):")
print("=" * 80)
for i, row in day_top3.iterrows():
    print(
        f"  #{i+1}: {row['algorithm']:20s} k={int(row['k'])}  "
        f"Sil={row['silhouette']:.4f}  DB={row['davies_bouldin']:.4f}  "
        f"CH={row['calinski_harabasz']:.1f}  "
        f"(combined_rank={int(row['combined_rank'])})"
    )

# Best configuration
best_day_algo = day_top3.iloc[0]["algorithm"]
best_day_k = int(day_top3.iloc[0]["k"])
print(f"\nBest TYPE_DAY: {best_day_algo} with k={best_day_k}")

---
## 5. TYPE_WEEK clustering (weekly profiles, 168h)

The same procedure as in Section 4 (same three algorithms, same validation indices, same combined-rank criterion) is applied to the 168-hour weekly profiles. Because there are far fewer weekly profiles than daily profiles, the sweep is limited to $k \in \{2, \dots, 5\}$.

In [ ]:
# ── TYPE_WEEK sweep ───────────────────────────────────────────────────────
WEEK_ALGORITHMS = ["kmeans_euclidean", "kshape", "kmeans_dtw"]
WEEK_K_RANGE = list(range(2, 6))  # 2..5

week_results = []
week_all_labels = {}
week_all_centroids = {}

total_configs_w = len(WEEK_ALGORITHMS) * len(WEEK_K_RANGE)
print(f"Running {total_configs_w} TYPE_WEEK configurations...\n")

for algo in WEEK_ALGORITHMS:
    for k in WEEK_K_RANGE:
        print(f"  {algo:20s} k={k} ...", end=" ", flush=True)
        try:
            labels, centroids, inertia = run_clustering(
                weekly_matrix, algo, k, random_seed=RANDOM_SEED
            )
            metrics = compute_clustering_metrics(weekly_matrix, labels)
            metrics["inertia"] = inertia
            metrics["algorithm"] = algo
            metrics["k"] = k
            metrics["clustering_type"] = "type_week"
            week_results.append(metrics)
            week_all_labels[(algo, k)] = labels
            week_all_centroids[(algo, k)] = centroids
            print(
                f"Sil={metrics['silhouette']:.4f}  "
                f"DB={metrics['davies_bouldin']:.4f}  "
                f"CH={metrics['calinski_harabasz']:.1f}"
            )
        except Exception as e:
            print(f"FAILED: {e}")
            week_results.append({
                "algorithm": algo, "k": k, "clustering_type": "type_week",
                "silhouette": np.nan, "davies_bouldin": np.nan,
                "calinski_harabasz": np.nan, "inertia": np.nan,
            })

week_results_df = pd.DataFrame(week_results)
print(f"\nCompleted {len(week_results)} configurations.")

In [ ]:
# ── TYPE_WEEK results table ───────────────────────────────────────────────
week_display = week_results_df[display_cols].copy()
week_display = week_display.sort_values(["algorithm", "k"]).reset_index(drop=True)

styled_w = week_display.style.format({
    "silhouette": "{:.4f}",
    "davies_bouldin": "{:.4f}",
    "calinski_harabasz": "{:.1f}",
    "inertia": "{:.1f}",
}).background_gradient(
    subset=["silhouette"], cmap="Greens"
).background_gradient(
    subset=["davies_bouldin"], cmap="Reds_r"
)

print("TYPE_WEEK clustering results:")
display(styled_w)

In [ ]:
# ── Automatic selection TYPE_WEEK ─────────────────────────────────────────
week_top3 = rank_clustering_configs(week_results_df, top_n=3)
print("TOP-3 TYPE_WEEK configurations (combined Silhouette + DB rank):")
print("=" * 80)
for i, row in week_top3.iterrows():
    print(
        f"  #{i+1}: {row['algorithm']:20s} k={int(row['k'])}  "
        f"Sil={row['silhouette']:.4f}  DB={row['davies_bouldin']:.4f}  "
        f"CH={row['calinski_harabasz']:.1f}  "
        f"(combined_rank={int(row['combined_rank'])})"
    )

best_week_algo = week_top3.iloc[0]["algorithm"]
best_week_k = int(week_top3.iloc[0]["k"])
print(f"\nBest TYPE_WEEK: {best_week_algo} with k={best_week_k}")

---
## 6. Combination of day type and week type (expert domains)

Each hourly observation receives:
- `cluster_day` — the label of its calendar day under the selected TYPE_DAY configuration;
- `cluster_week` — the label of its ISO week under the selected TYPE_WEEK configuration;
- `expert_id = f"day{cluster_day}_week{cluster_week}"`.

The total number of expert domains is therefore at most **N = k_day × k_week** (only combinations that actually occur are counted). Rare combinations (fewer than 50 hourly observations) are flagged, since they might require special treatment (e.g., merging with another domain).

### 6.1 Final choice of the configuration (manual override)

The next two cells adjust the automatic selection of Sections 4–5:

1. The TYPE_DAY top-3 table is re-sorted by its Davies–Bouldin rank (`rank_db`). This only changes the order of `day_top3`, which is used later for the saved top-3 summary and the final printout; the selected algorithm `best_day_algo` was already fixed in Section 4.
2. The number of clusters is **fixed manually** to $k_{day} = 3$ and $k_{week} = 2$, overriding the $k$ chosen by the combined rank. The algorithms remain those ranked first by the combined criterion (`best_day_algo`, `best_week_algo`), so the final labels are those obtained by these algorithms with $k_{day} = 3$ and $k_{week} = 2$ during the sweeps.

In [ ]:
# Re-order the TYPE_DAY top-3 table by Davies-Bouldin rank (order only; best_day_algo is unchanged)
day_top3 = day_top3.sort_values("rank_db").reset_index(drop=True)

In [ ]:
# Manual override of the number of clusters (see Section 6.1)
best_day_k=3
best_week_k=2

### 6.2 Hourly assignment and expert identifiers

The daily and weekly labels are mapped to every hourly observation: days are matched by calendar date and weeks by their ISO `year_week` key. Hours belonging to an incomplete day or an incomplete ISO week (which have no label) are dropped. The timestamp is kept as an explicit `datetime` column, because the merges discard the index and the downstream notebooks join the assignments to the features through this column.

In [ ]:
# ── Retrieve the labels of the selected configurations ──────────────────────────────────────────────────
best_day_labels = day_all_labels[(best_day_algo, best_day_k)]
best_week_labels = week_all_labels[(best_week_algo, best_week_k)]

# ── Build the daily assignment DataFrame ──────────────────────────────────────
day_assignment = pd.DataFrame({
    "date": daily_dates,
    "cluster_day": best_day_labels,
})
day_assignment["date"] = pd.to_datetime(day_assignment["date"])

# ── Build the weekly assignment DataFrame ───────────────────────────────────
week_assignment = pd.DataFrame({
    "year_week": weekly_labels,
    "week_start": weekly_dates,
    "cluster_week": best_week_labels,
})

# ── Map the assignments to every hourly observation ──────────────────────────────
df_assignments = df[[TARGET_COL]].copy()
# Keep the timestamp as a column: the merges below discard the index, and the
# downstream notebooks join the assignments to the features through this column.
df_assignments["datetime"] = df_assignments.index
df_assignments["date"] = df_assignments.index.normalize()
df_assignments["iso_year"] = df_assignments.index.isocalendar().year.values
df_assignments["iso_week"] = df_assignments.index.isocalendar().week.values
df_assignments["year_week"] = (
    df_assignments["iso_year"].astype(str) + "-W" +
    df_assignments["iso_week"].astype(str).str.zfill(2)
)

# Merge daily clusters
df_assignments = df_assignments.merge(
    day_assignment, on="date", how="left"
)

# Merge weekly clusters
df_assignments = df_assignments.merge(
    week_assignment[["year_week", "cluster_week"]], on="year_week", how="left"
)

# Drop observations without a cluster assignment (incomplete days/weeks)
n_before = len(df_assignments)
df_assignments = df_assignments.dropna(subset=["cluster_day", "cluster_week"])
df_assignments["cluster_day"] = df_assignments["cluster_day"].astype(int)
df_assignments["cluster_week"] = df_assignments["cluster_week"].astype(int)
n_after = len(df_assignments)

print(f"Observations with both assignments: {n_after}/{n_before} ({n_after/n_before*100:.1f}%)")

# ── Create expert_id ───────────────────────────────────────────────────────────
df_assignments["expert_id"] = (
    "day" + df_assignments["cluster_day"].astype(str) +
    "_week" + df_assignments["cluster_week"].astype(str)
)

n_experts = df_assignments["expert_id"].nunique()
print(f"\nTotal number of experts (k_day x k_week = {best_day_k} x {best_week_k}): {n_experts}")
print(f"\nFrequency per expert:")
print(df_assignments["expert_id"].value_counts().to_string())

In [ ]:
# ── Frequency table: type_day x type_week ─────────────────────────────────
freq_table = pd.crosstab(
    df_assignments["cluster_day"],
    df_assignments["cluster_week"],
    margins=True,
    margins_name="Total",
)
freq_table.index.name = "cluster_day"
freq_table.columns.name = "cluster_week"

print("Frequency table (hourly observations per type_day x type_week):")
display(freq_table)

# ── Identify rare combinations ─────────────────────────────────────────
MIN_OBS_THRESHOLD = 50
expert_counts = df_assignments["expert_id"].value_counts()
rare_experts = expert_counts[expert_counts < MIN_OBS_THRESHOLD]

if len(rare_experts) > 0:
    print(f"\nRare combinations (<{MIN_OBS_THRESHOLD} observations), flagged for special treatment:")
    for eid, count in rare_experts.items():
        print(f"  {eid}: {count} observations")
else:
    print(f"\nNo rare combinations (<{MIN_OBS_THRESHOLD} obs). All experts have sufficient data.")

print(f"\nSummary:")
print(f"  Selected daily configuration:  {best_day_algo} k={best_day_k}")
print(f"  Selected weekly configuration: {best_week_algo} k={best_week_k}")
print(f"  Total number of experts:       {n_experts}")
print(f"  Rare experts:                  {len(rare_experts)}")

---
## 7. Visualization

Figures documenting the selected clustering (all saved to `results/`):

1. **Mean daily load profile per `cluster_day`** — 24-hour mean curve with a ±1 standard deviation band (`03_clustering_day_profiles.png`).
2. **Mean weekly load profile per `cluster_week`** — 168-hour mean curve with a ±1 standard deviation band (`03_clustering_week_profiles.png`).
3. **Frequency heatmap of type_day × type_week** — number of hourly observations per expert domain; rare combinations (< 50 observations) are outlined in red (`03_clustering_expert_heatmap.png`).
4. **Calendar view of the `cluster_day` assignments** — one panel per year, ISO week × day of week (`03_clustering_day_calendar.png`).
5. **Temporal consistency** — Adjusted Rand Index (ARI) on sliding windows (`03_clustering_temporal_consistency.png`).
6. **Comparison of internal validation metrics** — Silhouette and Davies–Bouldin versus $k$ for every algorithm (`03_clustering_metrics_comparison.png`).

**Temporal consistency (item 5).** To assess whether the cluster structure is stable over time, the profiles inside a sliding window are re-clustered with the same algorithm and $k$ (with `max_iter = 30`), and the resulting labels are compared with the labels that the same profiles received in the full-period clustering through the ARI (which is invariant to label permutations). Windows are 90 days long for TYPE_DAY and 180 days long for TYPE_WEEK (fewer samples per window), with a 30-day step in both cases; windows with fewer than $5k$ profiles are skipped. Each ARI value is plotted at the centre of its window. The horizontal reference lines at ARI = 0.8 and 0.5 are visual guides only.

In [ ]:
# ── 7.1 Mean daily profile per cluster_day (mean ± 1 std) ─────────────────
fig, ax = plt.subplots(figsize=(12, 6))

for c in sorted(np.unique(best_day_labels)):
    mask = best_day_labels == c
    cluster_profiles = daily_matrix[mask]
    mean_profile = cluster_profiles.mean(axis=0)
    std_profile = cluster_profiles.std(axis=0)

    color = PALETTE[c % len(PALETTE)]
    ax.plot(
        range(24), mean_profile, color=color, linewidth=2,
        label=f"Cluster {c} (n={mask.sum()})"
    )
    ax.fill_between(
        range(24),
        mean_profile - std_profile,
        mean_profile + std_profile,
        alpha=0.15, color=color,
    )

ax.set_xlabel("Hour of day")
ax.set_ylabel("Demand (kWh)")
ax.set_title(
    f"TYPE_DAY Cluster Profiles ({best_day_algo}, k={best_day_k})"
)
ax.set_xticks(range(0, 24, 2))
ax.legend(loc="upper left", framealpha=0.9)
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "03_clustering_day_profiles.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
# ── 7.2 Mean weekly profile per cluster_week (mean ± 1 std) ──────────────
fig, ax = plt.subplots(figsize=(14, 6))

for c in sorted(np.unique(best_week_labels)):
    mask = best_week_labels == c
    cluster_profiles = weekly_matrix[mask]
    mean_profile = cluster_profiles.mean(axis=0)
    std_profile = cluster_profiles.std(axis=0)

    color = PALETTE[c % len(PALETTE)]
    ax.plot(
        range(168), mean_profile, color=color, linewidth=1.5,
        label=f"Cluster {c} (n={mask.sum()})"
    )
    ax.fill_between(
        range(168),
        mean_profile - std_profile,
        mean_profile + std_profile,
        alpha=0.12, color=color,
    )

# Day separators
for d in range(1, 7):
    ax.axvline(d * 24, color="gray", linestyle=":", alpha=0.4, linewidth=0.8)

ax.set_xlabel("Hour of week")
ax.set_ylabel("Demand (kWh)")
ax.set_title(
    f"TYPE_WEEK Cluster Profiles ({best_week_algo}, k={best_week_k})"
)
day_ticks = [i * 24 + 12 for i in range(7)]
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
ax.set_xticks(day_ticks)
ax.set_xticklabels(day_labels)
ax.legend(loc="upper left", framealpha=0.9)
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "03_clustering_week_profiles.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
# ── 7.3 Frequency heatmap type_day x type_week ───────────────────────────
freq_no_margins = pd.crosstab(
    df_assignments["cluster_day"],
    df_assignments["cluster_week"],
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    freq_no_margins,
    annot=True,
    fmt="d",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Number of hourly observations"},
)
ax.set_xlabel(f"cluster_week (k={best_week_k})")
ax.set_ylabel(f"cluster_day (k={best_day_k})")
ax.set_title("Expert Domain Frequency: type_day x type_week")

# Mark rare combinations
for i in range(freq_no_margins.shape[0]):
    for j in range(freq_no_margins.shape[1]):
        val = freq_no_margins.iloc[i, j]
        if val < MIN_OBS_THRESHOLD:
            ax.add_patch(plt.Rectangle(
                (j, i), 1, 1, fill=False, edgecolor="red", linewidth=2.5
            ))

plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "03_clustering_expert_heatmap.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Red borders indicate rare combinations (<50 observations).")

In [ ]:
# ── 7.4 Calendar view of the cluster_day assignments ──────────────────────
day_assign_for_cal = day_assignment.copy()
day_assign_for_cal["year"] = day_assign_for_cal["date"].dt.year
years = sorted(day_assign_for_cal["year"].unique())

n_years = len(years)
fig, axes = plt.subplots(
    n_years, 1,
    figsize=(16, 1.8 * n_years + 1),
    squeeze=False,
)

cmap = plt.cm.get_cmap("Set2", best_day_k)

for idx, year in enumerate(years):
    ax = axes[idx, 0]
    year_data = day_assign_for_cal[day_assign_for_cal["year"] == year].copy()
    year_data["day_of_year"] = year_data["date"].dt.dayofyear
    year_data["week_of_year"] = year_data["date"].dt.isocalendar().week.values.astype(int)
    year_data["day_of_week"] = year_data["date"].dt.dayofweek  # 0=Mon

    # Create a matrix: week x day_of_week
    pivot = year_data.pivot_table(
        index="week_of_year", columns="day_of_week",
        values="cluster_day", aggfunc="first"
    )

    sns.heatmap(
        pivot.T,
        ax=ax,
        cmap=cmap,
        vmin=0, vmax=best_day_k - 1,
        linewidths=0.3,
        linecolor="white",
        cbar=idx == 0,
        cbar_kws={"label": "cluster_day", "ticks": range(best_day_k)} if idx == 0 else {},
        xticklabels=5,
        yticklabels=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    )
    ax.set_title(f"{year}", fontsize=11, fontweight="bold")
    ax.set_xlabel("" if idx < n_years - 1 else "ISO week")
    ax.set_ylabel("")

fig.suptitle(
    f"Temporal Distribution of TYPE_DAY Clusters ({best_day_algo}, k={best_day_k})",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "03_clustering_day_calendar.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
# ── 7.5 Temporal consistency: ARI on sliding windows ──────────────────────

def compute_sliding_ari(
    dates: np.ndarray,
    labels: np.ndarray,
    profiles: np.ndarray,
    algorithm: str,
    k: int,
    window_days: int = 90,
    step_days: int = 30,
    random_seed: int = 42,
) -> Tuple[List, List]:
    """
    Compute ARI between full-dataset clustering and per-window re-clustering.
    Measures temporal stability of the clustering structure.
    """
    dates_pd = pd.to_datetime(dates)
    min_date = dates_pd.min()
    max_date = dates_pd.max()

    window_starts = []
    ari_values = []

    current_start = min_date
    while current_start + pd.Timedelta(days=window_days) <= max_date:
        window_end = current_start + pd.Timedelta(days=window_days)
        mask = (dates_pd >= current_start) & (dates_pd < window_end)

        if mask.sum() < k * 5:  # Need enough samples
            current_start += pd.Timedelta(days=step_days)
            continue

        window_profiles = profiles[mask]
        window_labels_full = labels[mask]  # from full-dataset clustering

        try:
            window_labels_new, _, _ = run_clustering(
                window_profiles, algorithm, k, random_seed=random_seed, max_iter=30
            )
            ari = adjusted_rand_score(window_labels_full, window_labels_new)
            window_starts.append(current_start + pd.Timedelta(days=window_days // 2))
            ari_values.append(ari)
        except Exception:
            pass

        current_start += pd.Timedelta(days=step_days)

    return window_starts, ari_values


# Compute ARI for TYPE_DAY
print("Computing temporal consistency (ARI) for TYPE_DAY...")
day_ari_dates, day_ari_values = compute_sliding_ari(
    daily_dates, best_day_labels, daily_matrix,
    best_day_algo, best_day_k,
    window_days=90, step_days=30,
)

# Compute ARI for TYPE_WEEK
print("Computing temporal consistency (ARI) for TYPE_WEEK...")
week_ari_dates, week_ari_values = compute_sliding_ari(
    weekly_dates, best_week_labels, weekly_matrix,
    best_week_algo, best_week_k,
    window_days=180, step_days=30,  # larger window for weekly (fewer samples)
)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

ax = axes[0]
ax.plot(day_ari_dates, day_ari_values, "o-", color="steelblue", markersize=4, linewidth=1.5)
ax.axhline(y=0.8, color="green", linestyle="--", alpha=0.5, label="ARI=0.8 (good)")
ax.axhline(y=0.5, color="orange", linestyle="--", alpha=0.5, label="ARI=0.5 (moderate)")
ax.set_ylabel("Adjusted Rand Index")
ax.set_title(
    f"Temporal Consistency — TYPE_DAY ({best_day_algo}, k={best_day_k})\n"
    f"ARI between full-dataset and 90-day window re-clustering"
)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="lower left")

ax = axes[1]
ax.plot(week_ari_dates, week_ari_values, "s-", color="darkorange", markersize=4, linewidth=1.5)
ax.axhline(y=0.8, color="green", linestyle="--", alpha=0.5, label="ARI=0.8 (good)")
ax.axhline(y=0.5, color="orange", linestyle="--", alpha=0.5, label="ARI=0.5 (moderate)")
ax.set_xlabel("Date")
ax.set_ylabel("Adjusted Rand Index")
ax.set_title(
    f"Temporal Consistency — TYPE_WEEK ({best_week_algo}, k={best_week_k})\n"
    f"ARI between full-dataset and 180-day window re-clustering"
)
ax.set_ylim(-0.05, 1.05)
ax.legend(loc="lower left")

plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "03_clustering_temporal_consistency.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

print(f"\nTYPE_DAY ARI:  mean={np.mean(day_ari_values):.3f}, min={np.min(day_ari_values):.3f}")
print(f"TYPE_WEEK ARI: mean={np.mean(week_ari_values):.3f}, min={np.min(week_ari_values):.3f}")

In [ ]:
# ── 7.6 Comparison of internal validation metrics ───────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# TYPE_DAY metrics
for algo in DAY_ALGORITHMS:
    subset = day_results_df[day_results_df["algorithm"] == algo]
    axes[0, 0].plot(subset["k"], subset["silhouette"], "o-", label=algo, markersize=5)
    axes[0, 1].plot(subset["k"], subset["davies_bouldin"], "o-", label=algo, markersize=5)

axes[0, 0].set_title("TYPE_DAY: Silhouette Score (higher = better)")
axes[0, 0].set_xlabel("k")
axes[0, 0].set_ylabel("Silhouette")
axes[0, 0].legend(fontsize=8)

axes[0, 1].set_title("TYPE_DAY: Davies-Bouldin Index (lower = better)")
axes[0, 1].set_xlabel("k")
axes[0, 1].set_ylabel("DB Index")
axes[0, 1].legend(fontsize=8)

# TYPE_WEEK metrics
for algo in WEEK_ALGORITHMS:
    subset = week_results_df[week_results_df["algorithm"] == algo]
    axes[1, 0].plot(subset["k"], subset["silhouette"], "o-", label=algo, markersize=5)
    axes[1, 1].plot(subset["k"], subset["davies_bouldin"], "o-", label=algo, markersize=5)

axes[1, 0].set_title("TYPE_WEEK: Silhouette Score (higher = better)")
axes[1, 0].set_xlabel("k")
axes[1, 0].set_ylabel("Silhouette")
axes[1, 0].legend(fontsize=8)

axes[1, 1].set_title("TYPE_WEEK: Davies-Bouldin Index (lower = better)")
axes[1, 1].set_xlabel("k")
axes[1, 1].set_ylabel("DB Index")
axes[1, 1].legend(fontsize=8)

plt.suptitle("Internal Validation Metrics by Algorithm and k", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(
    os.path.join(RESULTS_DIR, "03_clustering_metrics_comparison.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

---
## 8. Saving artifacts

All clustering artifacts are written to `data/clustering/` (`CLUSTERING_DIR`):

| File | Description |
|---------|------------|
| `cluster_assignments.parquet` | `datetime`, `cluster_day`, `cluster_week` and `expert_id` for every hour (input to Notebooks 04–08) |
| `centroids_day.pkl` | Centroids of the selected TYPE_DAY configuration, with its algorithm and $k$ (used by the distance-based gating in Notebook 06) |
| `centroids_week.pkl` | Centroids of the selected TYPE_WEEK configuration, with its algorithm and $k$ |
| `clustering_metrics.json` | All internal validation metrics per (algorithm, $k$) for both time scales |
| `top_configs.json` | Top-3 configurations per time scale and a summary of the selected combination |
| `top3_labels_archive.pkl` | Labels, centroids and dates/ISO weeks of the top-3 configurations per time scale |

The last three files are kept for documentation and reproducibility; they are not read by the subsequent notebooks of the pipeline.

Note that both centroid files store the metadata entry `"normalization": "z_normalize_per_row"`, although, as noted in Section 3, the clustering was fit on the raw (non-normalized) profiles.

In [ ]:
df_assignments.head(25)

In [ ]:
# ── 8.1 Save cluster assignments ──────────────────────────────────────────
assignments_save = df_assignments[["datetime", "cluster_day", "cluster_week", "expert_id"]].copy()
assignments_path = os.path.join(CLUSTERING_DIR, "cluster_assignments.parquet")
assignments_save.to_parquet(assignments_path)
print(f"Saved: {assignments_path}")
print(f"  Shape: {assignments_save.shape}")
print(f"  Columns: {list(assignments_save.columns)}")

# ── 8.2 Save centroids ────────────────────────────────────────────────────
best_day_centroids = day_all_centroids[(best_day_algo, best_day_k)]
best_week_centroids = week_all_centroids[(best_week_algo, best_week_k)]

centroids_day_path = os.path.join(CLUSTERING_DIR, "centroids_day.pkl")
with open(centroids_day_path, "wb") as f:
    pickle.dump({
        "centroids": best_day_centroids,
        "algorithm": best_day_algo,
        "k": best_day_k,
        "normalization": "z_normalize_per_row",
    }, f)
print(f"\nSaved: {centroids_day_path}")
print(f"  Centroid shape: {best_day_centroids.shape}")

centroids_week_path = os.path.join(CLUSTERING_DIR, "centroids_week.pkl")
with open(centroids_week_path, "wb") as f:
    pickle.dump({
        "centroids": best_week_centroids,
        "algorithm": best_week_algo,
        "k": best_week_k,
        "normalization": "z_normalize_per_row",
    }, f)
print(f"\nSaved: {centroids_week_path}")
print(f"  Centroid shape: {best_week_centroids.shape}")

In [ ]:
# ── 8.3 Save all clustering metrics ─────────────────────────────
all_metrics = pd.concat([day_results_df, week_results_df], ignore_index=True)
metrics_dict = all_metrics.to_dict(orient="records")

# Convert numpy types to native Python types for JSON serialization
def convert_numpy(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj) if not np.isnan(obj) else None
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

metrics_clean = []
for m in metrics_dict:
    metrics_clean.append({k: convert_numpy(v) for k, v in m.items()})

metrics_path = os.path.join(CLUSTERING_DIR, "clustering_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_clean, f, indent=2)
print(f"Saved: {metrics_path}")
print(f"  Total configurations: {len(metrics_clean)}")

# ── 8.4 Save the top-3 configurations (for reference) ──────
top_configs = {
    "type_day": [],
    "type_week": [],
    "best_combination": {
        "day_algorithm": best_day_algo,
        "day_k": best_day_k,
        "week_algorithm": best_week_algo,
        "week_k": best_week_k,
        "n_experts": n_experts,
        "n_rare_experts": len(rare_experts),
    },
}

for _, row in day_top3.iterrows():
    top_configs["type_day"].append({
        "algorithm": row["algorithm"],
        "k": int(row["k"]),
        "silhouette": float(row["silhouette"]),
        "davies_bouldin": float(row["davies_bouldin"]),
        "calinski_harabasz": float(row["calinski_harabasz"]),
        "combined_rank": int(row["combined_rank"]),
    })

for _, row in week_top3.iterrows():
    top_configs["type_week"].append({
        "algorithm": row["algorithm"],
        "k": int(row["k"]),
        "silhouette": float(row["silhouette"]),
        "davies_bouldin": float(row["davies_bouldin"]),
        "calinski_harabasz": float(row["calinski_harabasz"]),
        "combined_rank": int(row["combined_rank"]),
    })

top_configs_path = os.path.join(CLUSTERING_DIR, "top_configs.json")
with open(top_configs_path, "w") as f:
    json.dump(top_configs, f, indent=2)
print(f"\nSaved: {top_configs_path}")
print(f"  Top-3 TYPE_DAY configurations:  {[c['algorithm'] + '_k' + str(c['k']) for c in top_configs['type_day']]}")
print(f"  Top-3 TYPE_WEEK configurations: {[c['algorithm'] + '_k' + str(c['k']) for c in top_configs['type_week']]}")

In [ ]:
# ── 8.5 Save labels and centroids of the top-3 configurations (for reference) ─────
all_labels_archive = {
    "type_day": {},
    "type_week": {},
}

for _, row in day_top3.iterrows():
    key = f"{row['algorithm']}_k{int(row['k'])}"
    all_labels_archive["type_day"][key] = {
        "labels": day_all_labels[(row["algorithm"], int(row["k"]))].tolist(),
        "centroids": day_all_centroids[(row["algorithm"], int(row["k"]))].tolist(),
        "dates": [str(d) for d in daily_dates],
    }

for _, row in week_top3.iterrows():
    key = f"{row['algorithm']}_k{int(row['k'])}"
    all_labels_archive["type_week"][key] = {
        "labels": week_all_labels[(row["algorithm"], int(row["k"]))].tolist(),
        "centroids": week_all_centroids[(row["algorithm"], int(row["k"]))].tolist(),
        "year_weeks": weekly_labels,
    }

archive_path = os.path.join(CLUSTERING_DIR, "top3_labels_archive.pkl")
with open(archive_path, "wb") as f:
    pickle.dump(all_labels_archive, f)
print(f"Saved: {archive_path}")
print(f"  TYPE_DAY configurations:  {list(all_labels_archive['type_day'].keys())}")
print(f"  TYPE_WEEK configurations: {list(all_labels_archive['type_week'].keys())}")

---
## 9. Summary

The next cell prints a summary of the dual clustering pipeline: profile construction, the TYPE_DAY and TYPE_WEEK sweeps and selections, the resulting expert domains, the temporal consistency (ARI) and the list of saved artifacts.

Note that the Silhouette and Davies–Bouldin values printed for each time scale are those of the **first row of the corresponding top-3 table**. Because of the manual override of $k$ in Section 6.1 (and the re-sorting of the TYPE_DAY table), this row is not necessarily the configuration finally used to build the expert domains.

In [ ]:
print("=" * 80)
print("NOTEBOOK 03 — DUAL CLUSTERING SUMMARY")
print("=" * 80)
print(f"\n1. PROFILE CONSTRUCTION")
print(f"   Daily profiles:  {daily_matrix.shape} (raw; row-wise z-normalized copies also computed)")
print(f"   Weekly profiles: {weekly_matrix.shape} (raw; row-wise z-normalized copies also computed)")
print(f"\n2. TYPE_DAY CLUSTERING")
print(f"   Algorithms tested: {DAY_ALGORITHMS}")
print(f"   Range of k: {DAY_K_RANGE}")
print(f"   Selected: {best_day_algo} with k={best_day_k}")
print(f"   First row of the top-3 table: Silhouette={day_top3.iloc[0]['silhouette']:.4f}, DB={day_top3.iloc[0]['davies_bouldin']:.4f}")
print(f"\n3. TYPE_WEEK CLUSTERING")
print(f"   Algorithms tested: {WEEK_ALGORITHMS}")
print(f"   Range of k: {WEEK_K_RANGE}")
print(f"   Selected: {best_week_algo} with k={best_week_k}")
print(f"   First row of the top-3 table: Silhouette={week_top3.iloc[0]['silhouette']:.4f}, DB={week_top3.iloc[0]['davies_bouldin']:.4f}")
print(f"\n4. EXPERT DOMAINS")
print(f"   N experts = {best_day_k} x {best_week_k} = {n_experts}")
print(f"   Rare combinations (<{MIN_OBS_THRESHOLD} obs): {len(rare_experts)}")
print(f"   Total hourly observations assigned: {len(df_assignments)}")
print(f"\n5. TEMPORAL CONSISTENCY")
if len(day_ari_values) > 0:
    print(f"   TYPE_DAY ARI:  mean={np.mean(day_ari_values):.3f}, min={np.min(day_ari_values):.3f}")
if len(week_ari_values) > 0:
    print(f"   TYPE_WEEK ARI: mean={np.mean(week_ari_values):.3f}, min={np.min(week_ari_values):.3f}")
print(f"\n6. SAVED ARTIFACTS")
print(f"   {assignments_path}")
print(f"   {centroids_day_path}")
print(f"   {centroids_week_path}")
print(f"   {metrics_path}")
print(f"   {top_configs_path}")
print(f"   {archive_path}")
print("\n" + "=" * 80)
print("Proceed to Notebook 04 (cluster analysis and cluster classifiers).")
print("Top-3 configurations saved for reference in top_configs.json and top3_labels_archive.pkl.")
print("=" * 80)